# PV feed-in and grid draw in example district scenarios

This notebook investigates example district scenarios and their **hourly PV production and grid interaction**.

The purpose is to assess the potential strain on the electricity grid from **PV feed-in**, and to compare it with the **grid draw** expected in the same district scenarios. The analysis focuses on:

- annual PV self-consumption and self-sufficiency indicators,
- hourly magnitude and seasonal timing of grid draw and PV feed-in,
- peak power,
- and the duration of high feed-in and grid-draw events.

Each Excel file is a single-scenario PEExcel timestep export (`IN.STAGE`, `OUT.Live`, `SIM2`). PEXL loads all files and combines their scenarios into one `Project`.

In [ ]:
from pathlib import Path

import pandas as pd
import pexl

# Directory containing the single-scenario timestep exports.
folder = Path(r"C:\Users\schneids\OneDrive\Documents")
file_pattern = "Aspern PVCheck*.xlsx"

# Detailed view of the first hours of each duration curve.
zoom_hours = 100

flow_colors = {
    "Grid draw": "firebrick",
    "PV feed-in": "forestgreen",
}

## Load all timestep scenarios

`Project.from_files()` combines the scenarios from the individual export files into one project. Scenario identity is retained by PEXL, while each scenario also carries its own hourly timestep data.

In [ ]:
files = [
    file
    for file in sorted(folder.glob(file_pattern))
    if not file.name.startswith("~$")
]

if not files:
    raise FileNotFoundError(
        f"No files matching {file_pattern!r} found in {folder}"
    )

project = pexl.Project.from_files(files)

missing_timeseries = [
    scenario.column_name
    for scenario in project
    if scenario.timeseries is None
]
if missing_timeseries:
    raise ValueError(
        f"Missing timestep data for scenarios: {missing_timeseries}"
    )

project

## Annual PV indicators

The PEXL project view is used to select only the annual indicators relevant to PV utilization and grid feed-in. This provides a compact comparison across all scenarios without manually accessing each scenario object.

In [ ]:
pv_metrics = project.out.select(
    "PV_own_consumption_direct",
    "PV_own_consumption_flex",
    "PV_own_consumption",
    "EUI_self_sufficiency",
    "PV_peak_grid_feedin",
    "PV_peak_grid_feedin_date",
)

pv_summary = (
    pd.DataFrame(pv_metrics.to_records())
    .set_index("scenario")
)

pv_summary

## Hourly grid interaction

The exported timestep values are specific energy quantities in Wh/m² per hourly timestep. Multiplication by the scenario gross floor area (`GFA_total`) and division by 1000 converts them to kWh per hour, numerically equivalent to the mean **kW** during that hour.

Two comparable datasets are prepared for every scenario:

- **Grid draw**: electricity drawn from the grid.
- **PV feed-in**: PV electricity exported to the grid.

The same data are also arranged as a nested mapping for direct multi-scenario plotting with `pexl.plot.duration_curve()`.

In [ ]:
grid_draw = {}
pv_feedin = {}
flow_data = {}

for scenario in project:
    df = scenario.timeseries
    factor = scenario.v.GFA_total / 1000

    grid = df["E_grid"] * factor
    feedin = df["PV_to_Egrid"] * factor

    grid_draw[scenario.column_name] = grid
    pv_feedin[scenario.column_name] = feedin

    flow_data[scenario.column_name] = {
        "Grid draw": grid,
        "PV feed-in": feedin,
    }

grid_draw = pd.DataFrame(grid_draw)
pv_feedin = pd.DataFrame(pv_feedin)

## Hourly and seasonal patterns

Heatmaps show **when** high grid interaction occurs during the year. Grid draw and PV feed-in are plotted separately so that each can use a suitable sequential color scale.

In [ ]:
pexl.plot.heatmap(
    grid_draw,
    unit="kW",
    layout="vertical",
    cmap="Reds",
)

pexl.plot.heatmap(
    pv_feedin,
    unit="kW",
    layout="vertical",
    cmap="Greens",
)

## Duration curves: magnitude and persistence of grid strain

The annual duration curves sort all hourly values from highest to lowest. They therefore show both:

- the **magnitude** of peak grid draw and PV feed-in,
- and the **number of hours** for which high grid loads persist.

Each row represents one district scenario. The second panel zooms into the first `zoom_hours` hours so that the most grid-relevant peak events can be compared more closely.

In [ ]:
fig, axes = pexl.plot.duration_curve(
    flow_data,
    unit="kW",
    layout="vertical",
    zoom_hours=zoom_hours,
    colors=flow_colors,
)

fig.suptitle(
    "Annual duration curves of grid draw and PV feed-in",
    y=1.01,
)

## Interpretation

The heatmaps and duration curves complement the annual indicators:

- **Peak magnitude** indicates the maximum local power that may affect connection or grid-capacity requirements.
- **Duration at high power** indicates whether these peaks are isolated events or persistent operating conditions.
- **PV feed-in versus grid draw** shows whether export peaks are of a similar, smaller, or larger magnitude than the district's expected import peaks.
- **Seasonal and hourly timing** helps distinguish PV-driven daytime/summer export stress from demand-driven grid draw.

The analysis is intentionally based on hourly district energy balances. It characterizes expected grid-facing power profiles, but does not replace a detailed electrical network calculation with voltage, line-loading, transformer, or power-flow constraints.